# Unidad 6: Despliegue (Deployment) e Integración Continua Básica
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## Introducción y Contexto de Negocio

Has construido un backend con base de datos, has aprendido a integrar APIs y a automatizar procesos. Pero, hasta ahora, todo corre en tu computadora local o en este cuaderno interactivo. Si querés que los usuarios usen tu producto digital, o que una app móvil consuma tus endpoints, tu código debe estar **online las 24 horas del día, los 7 días de la semana**.

El **Despliegue (Deployment)** es el proceso de publicar tu aplicación en servidores en la nube. Para asegurar que este proceso sea rápido, repetible y libre de errores humanos, aplicamos:
1. **Control de Versiones (Git/GitHub)**: Para colaborar en equipo y versionar la base de código.
2. **Contenedores (Docker)**: Para "empaquetar" la aplicación con todas sus librerías de forma que corra igual en cualquier servidor del mundo.
3. **CI/CD (Integración y Despliegue Continuo)**: Para automatizar las pruebas y la publicación online del software cada vez que actualizamos el código.

### Objetivos de Aprendizaje:
1. Diseñar la estructura estándar de archivos de un repositorio profesional en GitHub.
2. Escribir un archivo **Dockerfile** para contenedorizar una API FastAPI.
3. Desplegar servicios web en la nube usando plataformas PaaS (Render / Railway / Hugging Face).
4. Configurar un flujo básico de Integración Continua (CI) mediante **GitHub Actions**.


## 1. Estructura Estándar de un Proyecto en GitHub

Cuando creamos un repositorio en GitHub para producción, organizamos los archivos para facilitar el despliegue automático y las pruebas:

```text
mi-proyecto-digital/
│
├── .github/
│   └── workflows/
│       └── ci.yml             # Configuración de GitHub Actions (CI/CD)
│
├── app/
│   ├── __init__.py
│   ├── main.py                # Servidor FastAPI
│   └── database.py            # Conexión SQL/ORM
│
├── tests/
│   ├── __init__.py
│   └── test_main.py           # Pruebas unitarias (pytest)
│
├── .gitignore                 # Archivos que Git debe ignorar (ej. venv, .env, .db)
├── Dockerfile                 # Receta para empaquetar el contenedor Docker
├── requirements.txt           # Librerías necesarias para correr el proyecto
└── README.md                  # Documentación del proyecto (instructivo de uso)
```


## 2. Contenedorización con Docker

¿Alguna vez te pasó que un código corría bien en tu computadora pero fallaba en la de un compañero porque él tenía otra versión de Python u otro sistema operativo?

**Docker** resuelve este problema ("It works on my machine"). Empaqueta tu aplicación junto con sus dependencias y sistema operativo base en una unidad aislada llamada **Contenedor**.

Para crear un contenedor, escribimos un archivo de texto plano llamado `Dockerfile` (la "receta").

### Ejemplo de un Dockerfile para FastAPI:


In [ ]:
# Escribimos el Dockerfile de muestra en el entorno local de Colab
dockerfile_content = """
# 1. Imagen base oficial de Python (liviana)
FROM python:3.11-slim

# 2. Definir el directorio de trabajo dentro del contenedor
WORKDIR /app

# 3. Copiar la lista de requerimientos e instalar dependencias
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 4. Copiar todo el código de nuestra app al contenedor
COPY . .

# 5. Indicar el puerto en el que escuchará el contenedor
EXPOSE 8000

# 6. Comando para iniciar el servidor FastAPI al arrancar el contenedor
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content.strip())

print("Dockerfile creado localmente.")


## 3. Despliegue en la Nube (PaaS)

Para proyectos y prototipos de negocios digitales, no es necesario configurar servidores virtuales complejos en AWS o Google Cloud. Usamos plataformas **PaaS (Platform as a Service)** como **Render** o **Railway**, que se conectan a tu GitHub y realizan el despliegue automáticamente cada vez que hacés un `git push`.

### Checklist de Despliegue de tu API en Render:
1. **Crear cuenta**: Regístrate en [Render.com](https://render.com/) asociando tu cuenta de GitHub.
2. **Crear Web Service**: Elige "New > Web Service".
3. **Vincular Repositorio**: Selecciona el repositorio de GitHub de tu proyecto.
4. **Configuración de Ejecución**:
   - **Environment (Entorno)**: `Python` (o `Docker` si usas el Dockerfile).
   - **Build Command**: `pip install -r requirements.txt` (si no usas Docker).
   - **Start Command**: `uvicorn app.main:app --host 0.0.0.0 --port $PORT` (si no usas Docker).
5. **Click en Deploy**: Render creará una URL pública gratuita (ej. `https://mi-api.onrender.com`) y tu API estará lista para ser consumida.


## 4. Integración Continua (CI) con GitHub Actions

La **Integración Continua (CI)** consiste en ejecutar pruebas automáticas cada vez que un miembro del equipo sube código al repositorio para asegurar que los cambios no rompan la aplicación.

En GitHub esto se configura mediante **GitHub Actions**, creando un archivo de configuración YAML en la ruta `.github/workflows/ci.yml`.

### Ejemplo de Configuración de CI (`ci.yml`):


In [ ]:
# Escribimos el archivo de CI de muestra
ci_content = """
name: Pipeline de Integración Continua (CI)

on:
  push:
    branches: [ main, dev ]
  pull_request:
    branches: [ main ]

jobs:
  pruebas:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar Código del Repositorio
      uses: actions/checkout@v3

    - name: Configurar Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.11'

    - name: Instalar Dependencias
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements.txt
        pip install pytest ruff

    - name: Ejecutar Linter (Revisión de estilo PEP 8 con Ruff)
      run: ruff check app/

    - name: Ejecutar Pruebas Unitarias
      run: pytest
"""

import os
os.makedirs(".github/workflows", exist_ok=True)
with open(".github/workflows/ci.yml", "w") as f:
    f.write(ci_content.strip())

print("Archivo de integración continua .github/workflows/ci.yml creado.")


---

## Desafío Práctico (Trabajo Práctico 6)

Este desafío final te servirá como plantilla de inicio para tu **Proyecto Integrador de Cursada**.

**Consigna:**
1. Crear localmente los siguientes archivos del proyecto para simular la estructura oficial:
   - Un archivo `requirements.txt` con las librerías: `fastapi`, `pydantic`, `uvicorn`, `pytest`, `httpx` y `ruff`.
   - Una carpeta `app` y un archivo `app/main.py` con una API de FastAPI extremadamente sencilla que tenga un endpoint GET `/salud` y retorne `{"status": "ok"}`.
   - Una carpeta `tests` y un archivo `tests/test_main.py` que importe `TestClient` y valide que la ruta `/salud` devuelva código `200` y el JSON esperado.
2. Escribir un script de Python en el notebook que ejecute `pytest` localmente en la máquina del Colab para validar que el test pasa con éxito.
3. Explicar en una celda de texto detallada los pasos que seguirías en Render para subir esta estructura de archivos usando la opción de **Docker** en lugar de Python directo (revisando cómo tu Dockerfile copiaría la estructura creada).


In [ ]:
# --- Escribí tu resolución acá ---

# 1. Generar la estructura de archivos en Colab
# %%writefile requirements.txt
# ...
# %%writefile app/main.py
# ...
# %%writefile tests/test_main.py
# ...

# 2. Ejecutar pytest en el notebook
# Podés usar el comando de terminal: !pytest
# ...

# 3. Explicación de los pasos en Render usando Docker
# ...
